In [4]:
import torch
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

In [10]:
from model import FEATURE_COLUMNS

# Load only the 9 features expected by the model architecture
X_test_df = pd.read_csv("test_dataset.csv")[FEATURE_COLUMNS].apply(pd.to_numeric, errors='coerce').fillna(0)
y_test_df = pd.read_csv("test_labels.csv").apply(pd.to_numeric, errors='coerce').fillna(0)

# Convert to PyTorch tensors
X_test = torch.tensor(X_test_df.values, dtype=torch.float32)
y_test = torch.tensor(y_test_df.values.squeeze(), dtype=torch.float32)

print("Filtered X_test shape:", X_test.shape)

Filtered X_test shape: torch.Size([1000, 9])


In [8]:
print("X_test shape:", X_test.shape)

X_test shape: torch.Size([1000, 12])


In [13]:
y_true = pd.read_csv("test_labels.csv").iloc[:, -1].values.astype(int)   # last column = label

results = {}
for name, path in model_files.items():
    model = MythicThreatClassifier()
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval()

    with torch.no_grad():
        outputs = model(X_test).squeeze()
    print(name, "output range:", outputs.min().item(), outputs.max().item())

    probs = outputs  # change to torch.sigmoid(outputs) if model.py's forward returns logits
    preds = (probs >= 0.50).int().numpy()

    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    results[name] = {
        "accuracy": accuracy_score(y_true, preds),
        "recall": recall_score(y_true, preds, zero_division=0),
        "FN": fn, "FP": fp, "TP": tp, "TN": tn,
    }

print(pd.DataFrame(results).T)

ARES output range: -1.6121599674224854 1.6033899784088135
ATHENA output range: -2.467829942703247 2.379770040512085
APOLLO output range: -3.485909938812256 3.413439989089966
        accuracy  recall    FN    FP    TP     TN
ARES        0.95    0.95   5.0  45.0  95.0  855.0
ATHENA      0.95    0.75  25.0  25.0  75.0  875.0
APOLLO      0.95    0.55  45.0   5.0  55.0  895.0


In [16]:
best_model = min(results, key=lambda k: (results[k]["FN"], results[k]["FP"]))
print(f"Recommended Oracle: {best_model}")
print(f"FLAG: SIGMOID_{best_model}")

Recommended Oracle: ARES
FLAG: SIGMOID_ARES
